# Semantic Scope Extraction Demo
This notebook demonstrates and verifies the Tree-sitter-based semantic context extraction logic in ResponseIQ.

In [ ]:
import pytest
import asyncio
from pathlib import Path
import tempfile
import os
import sys

# Add src to path
sys.path.append(os.path.abspath("src"))

from responseiq.utils.context_extractor import extract_context_from_log, read_code_around_line

async def run_verification():
    with tempfile.TemporaryDirectory() as tmpdirname:
        temp_workspace = Path(tmpdirname)
        
        # Create a dummy python file with a function and a class
        # Add some padding to differentiate window vs scope
        file_content = """
class MyHelper:
    def __init__(self):
        self.value = 10
        self.attr = 20

# Padding line 1
# Padding line 2

def complex_operation(x):
    y = x + 1
    # This is the target line
    if y > 10:
        return y * 2
    return y
"""
        file_path = temp_workspace / "semantic_test.py"
        file_path.write_text(file_content.strip())
        
        # Line numbers:
        # 1: class MyHelper:
        # 2:     def __init__(self):
        # 3:         self.value = 10
        # 4:         self.attr = 20
        # 5: 
        # 6: # Padding line 1
        # 7: # Padding line 2
        # 8: 
        # 9: def complex_operation(x):
        # 10:    y = x + 1
        # 11:    # This is the target line
        
        target_line = 11
        
        # With simple window (5 lines context), range would be [11-5, 11+5] = [6, 16]
        # Line 6 is "# Padding line 1".
        # Line 9 is "def complex_operation(x):".
        
        # If semantic scope works, it should ONLY return lines 9-14 (the function body).
        # It should NOT return line 6 or 7.
        
        log_text = f'File "{file_path.name}", line {target_line}, in complex_operation'
        context = await extract_context_from_log(log_text, root_path=temp_workspace)
        
        print("--- Context Result ---")
        print(context)
        
        if "# Padding line 1" in context:
            print("\nFAIL: Padding line found (likely used simple window)")
        else:
            print("\nPASS: Padding line NOT found")
            
        if "def complex_operation(x):" in context:
            print("PASS: Function definition found")
        else:
            print("FAIL: Function definition NOT found")

await run_verification()